In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from statsmodels.stats.proportion import proportions_ztest
from scipy.stats import chi2_contingency

print("Imported successfully")

In [ ]:
# load dataset
df = pd.read_csv(r"D:\Project_programming\case_study_mc\Data\processesed\hse2_data.csv")
print("Loaded successfully")

### 1. COMPUTING THE TWO PROPORTION Z-TESTS

Let:
- \(P_{1999}\) = population prevalence of current smoking in 1999.
- \(P_{2019}\) = population prevalence of current smoking in 2019.

Null Hypothesis:

H_0:P_{1999} = P_{2019}

>There was no change in smoking prevalence between 1999 and 2019.

Alternate Hypothesis:

H_1:P_{1999}>P_{2019}

>Smoking prevalence decreased between 1999 and 2019.


In [ ]:
# calculate smoking prevalence
from numpy import absolute


smoking_summary = (
    df.groupby("year")
       .agg(
           sample_size=("current_smoker", "count"),
           smokers=("current_smoker", "sum"),
           prevalence=("current_smoker", "mean")
       )
)

smoking_summary["prevalence"] *= 100
# calculate percentage
p99 = np.array(smoking_summary.loc[1999,'prevalence'])
p19 = np.array(smoking_summary.loc[2019,'prevalence'])

absolute_change = np.subtract(p99,p19)
relative_change = np.divide(absolute_change,p99)*100

In [ ]:
# Number of smokers in each year
smokers = np.array([
  int(df.loc[df['year'] == 1999, 'current_smoker'].sum()),
  int(df.loc[df['year'] == 2019, 'current_smoker'].sum())
])
# valid sample size 
sample_size = np.array([
  int(df.loc[df['year'] == 1999, 'current_smoker'].count()),
  int(df.loc[df['year'] == 2019, 'current_smoker'].count())
])

# compute the z-tests
z_tests, p_value = proportions_ztest(
  count = smokers,
  nobs = sample_size,
  value = 0,
  alternative = 'larger'
)
# Summary statistics report
alpha = 0.05
if alpha > p_value:
  print("Reject the Null Hypothesis")
else:
  print("Fail to reject the Null Hypothesis")
print ("Z-statistics:", z_tests.round(5))
print("P-value:", p_value)
print("\n1999 prevalence:", p99.round(2))
print("2019 prevalence:", p19.round(2))
print("Absolute change (percentage points):", absolute_change.round(2))
print("Relative change (%):", relative_change.round(2))

### Interpretation of results

The results indicate a statistically significant decrease in smoking prevalence in England between 1999 and 2019. In 1999, the prevalence of smoking was 22.47%, compared with 15.39% in 2019. This represents an absolute reduction of 7.08 percentage points and a relative reduction of 31.52% over the 20-year period.

The two-proportion z-test produced a z-statistic of 11.06 and a p-value of 9.33 × 10⁻²⁹ (p < 0.001). Since the p-value is substantially below the conventional significance level of 0.05, the null hypothesis that smoking prevalence was equal in the two years can be rejected. There is therefore strong statistical evidence of a difference in smoking prevalence between 1999 and 2019, with the observed prevalence being lower in 2019.

Overall, these findings suggest that smoking prevalence in England declined considerably over the period studied. The relative reduction of approximately 31.5% indicates that smoking became less common compared with its prevalence in 1999.

However, the results should be interpreted with some caution. The HSE 1999 and HSE 2019 are independent cross-sectional surveys, so the findings demonstrate a change in population prevalence rather than a reduction in smoking among the same individuals. In addition, if the z-test was based on unweighted proportions, it may not account for the surveys' complex sampling designs. A stronger conclusion should therefore be supported by survey-weighted estimates and, ideally, age-standardised prevalence comparisons.

### 2. Test of Independence; Chi-square test
Tests whether smoking status is associated with survey year


>H_0: Smoking status is independent of year

>H_1: Smoking status is associated with year



In [ ]:
contigency = pd.crosstab(
  df['year'],
  df['current_smoker']
)
chi2, p, dof, expected = chi2_contingency(contigency)
print("Chi-square statistics:", chi2.round(5))
print("P-value:", p)
print("Degrees of freedom:", dof)

if alpha > p:
  print("Reject the Null hypothesis")
else: 
  print("Fail to reject the Null hypothesis")

### Interpretation of the chi-square test of independence

A chi-square test of independence was conducted to examine whether there was an association between smoking status and survey year (1999 and 2019).

The test produced a chi-square statistic (χ²(1)) of 121.95, with 1 degree of freedom and a p-value of 2.36 × 10⁻²⁸ (p < 0.001). Since the p-value is considerably smaller than the 0.05 significance level, the null hypothesis of no association between smoking status and survey year is rejected.

This indicates a statistically significant association between smoking status and survey year. In other words, the distribution of smoking status differed significantly between the 1999 and 2019 surveys. This provides evidence that smoking patterns changed over the period.

The direction of this association can be understood by examining the prevalence figures: current smoking decreased from 22.47% in 1999 to 15.39% in 2019. Therefore, the significant association is consistent with a decline in the proportion of current smokers over the 20-year period.

### 3. Gender x Year logistic regression

Test whether the decline in smoking prevalence differed by gender


Smoking ~ (Year + Gender) + (Year * Gender)

>H_0: The change in smoking prevalence between 1999 and 2019 does not differ by gender

>H_1: The change in smoking prevalence between 1999 and 2019 differs by gender

In [ ]:
# put year as a categorical variable
df['year'] = df['year'].astype(str)

# remove NaN values and convert the current smoker to int
df = df.dropna(subset = ['current_smoker']).copy()
df['current_smoker'] = df['current_smoker'].astype(int)

# Logistic regression with gender interaction

gender_model = smf.logit(
  'current_smoker ~ C(year)* C(sex)',
  data = df
).fit()
print(gender_model.summary())